# Taller 2 — Notebook 07: Generación de valor

**MINE-4101 · Predicción de churn para FinAlpes** · **Integrantes:** Daniel Triviño · Nicolás Bedoya

Entregable 5. Traduce los aciertos y errores del modelo a dinero: **¿cuánto gana un banco promedio por cada cliente
que el modelo evalúa, y en cuántos meses recupera la inversión?** Sigue el método del curso:

- Diapositivas *Métricas avanzadas y generación de valor* (sección "Medición de impacto", diap. 12–29): ROI, ganancia
  ajustada por predicción, valor esperado con la fórmula de Provost y punto de equilibrio.
- *Práctica 4, actividad 3* del sitio del curso
  ([PracticasMachineLearning](https://datascience-uniandes.github.io/PracticasMachineLearning/talleres/practica-4/actividad-3/))
  y la ayuda [Calcular el impacto de un modelo](https://datascience-uniandes.github.io/PracticasMachineLearning/ayudas/impacto-negocio/),
  que usan este mismo conjunto de datos.
- Provost, F. y Fawcett, T. (2013). *Data Science for Business*, cap. 7.

Pasos: (1) supuestos, (2) matriz de confusión del modelo, (3) valor esperado por cliente, (4) punto de equilibrio,
(5) sensibilidad a los supuestos, (6) umbral de decisión según la ganancia, (7) limitaciones.

> Ejecutar después de `04_modelado` (y preferiblemente del `05`).

In [ ]:
import json
import os

import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.metrics import confusion_matrix, precision_recall_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
sns.set_theme(style='whitegrid')

RUTA_PROCESADOS = '../data/processed'
RUTA_MODELOS = f'{RUTA_PROCESADOS}/modelos'
RUTA_FIGS = '../figs'
os.makedirs(RUTA_FIGS, exist_ok=True)

SEMILLA = 42   # la misma del 04 y el 05

## 1. Supuestos

El enunciado fija el banco "promedio" al que se le calcula el valor. Lo que no fija (costo por contacto y costo de
implementación) se toma del material del curso. Cada supuesto queda en una constante para poder cambiarlo y volver a
ejecutar.

| Supuesto | Valor | Fuente |
|---|---|---|
| Clientes del banco | 20.000 | Enunciado |
| Tasa anual de abandono | 15 % | Enunciado (nota 2: deserción promedio en banca minorista) |
| Éxito de la oferta *r* | 30 % de los que iban a abandonar | Enunciado |
| Ingreso anual por cliente *V* | COP 500.000–900.000 (central: 700.000) | Enunciado (nota 3: reporte 6-K de Nu, mayo de 2026) |
| Costo por cliente contactado *c* | *V*/30 | Ejercicio de churn de las diapositivas (diap. 28): la oferta de retención cuesta el doble de la retención normal (2 × 25 = 50) frente a un valor del cliente de 1.500 |
| Costo-hora del equipo | COP 28.700 | Diapositivas (diap. 20): costo-hora de un científico de datos en Colombia |
| Esfuerzo y rubros de la inversión | 9,5 personas-mes, otros rubros y 10 % de contingencia | Presupuesto de la Práctica 4, act. 3 (79.750 €) |

Se supone, como en la práctica, que la tasa de abandono es anual, que cada cliente se evalúa una vez al año y que *V*
es el ingreso de un año: retener a un cliente vale un año de ingreso.

In [ ]:
N_CLIENTES = 20_000
TASA_CHURN = 0.15                      # p(p): prior del banco, no la del dataset (~20 %)
R_EXITO = 0.30                         # r
V_RANGO = (500_000, 700_000, 900_000)  # ingreso anual por cliente (bajo, central, alto)
V = V_RANGO[1]
C_SOBRE_V = 50 / 1_500                 # diap. 28: oferta de 50 frente a un cliente que vale 1.500
C = V * C_SOBRE_V                      # costo por cliente contactado

# Inversión: estructura del presupuesto de la Práctica 4 (act. 3), con el costo-hora de la diap. 20
COSTO_HORA = 28_700
HORAS_MES = 160                        # 4 semanas de 40 h (la diap. 20 cuenta 2 semanas = 80 h)
personas_mes = {'Científico de datos': 3, 'Ingeniero de datos': 3, 'Ingeniero de ML': 2,
                'Líder de negocio (50 %)': 1.5}
PROP_OTROS = (4_500 + 8_000 + 3_000) / (18_000 + 16_500 + 12_000 + 10_500)   # nube + CRM + capacitación / personas

costo_personas = sum(personas_mes.values()) * HORAS_MES * COSTO_HORA
costo_otros = PROP_OTROS * costo_personas
INVERSION = (costo_personas + costo_otros) * 1.10                           # + 10 % de contingencia

print(f'V = COP {V:,.0f}/año  |  c = COP {C:,.0f} por contacto  |  r·V = COP {R_EXITO * V:,.0f}')
print(f'Inversión: personas COP {costo_personas:,.0f} + otros rubros COP {costo_otros:,.0f} '
      f'({PROP_OTROS:.0%} de personas) + 10 % = COP {INVERSION:,.0f}')

> **Por validar:** *c* y la inversión salen de ejemplos del curso, no de datos del mercado colombiano. La sección 5
> muestra cuánto cambian los resultados si se equivocan.

## 2. Matriz de confusión del modelo

Se usa el modelo elegido en el `04` (XGBoost) con el mismo umbral del `05` (máximo F1 en validación cruzada) y su
matriz de confusión en prueba. De ella solo se toman las **tasas por fila**: la sensibilidad (TPR, la proporción de los
que abandonan que el modelo marca) y la tasa de falsas alarmas (FPR, la proporción de los que se quedan que el modelo
marca). La precisión no se usa directamente porque depende de la proporción de abandonos, que en el banco (15 %) es
menor que en el dataset (\~20 %).

In [ ]:
df = pd.read_parquet(f'{RUTA_PROCESADOS}/churn_particion.parquet')
with open(f'{RUTA_MODELOS}/seleccion.json', encoding='utf-8') as f:
    seleccion = json.load(f)
ELEGIDO = seleccion['modelo_elegido']
modelo = joblib.load(f"{RUTA_MODELOS}/{seleccion['archivos'][ELEGIDO]}")

X = df[seleccion['variables_modelo']].astype(float)
y = df['churn']
es_prueba = (df['particion'] == 'prueba').to_numpy()
X_train, y_train, X_test, y_test = X[~es_prueba], y[~es_prueba], X[es_prueba], y[es_prueba]

# Umbral del 05: máximo F1 sobre predicciones de validación cruzada en entrenamiento
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
prob_cv = cross_val_predict(clone(modelo), X_train, y_train, cv=cv, method='predict_proba')[:, 1]
precision, recall, umbrales = precision_recall_curve(y_train, prob_cv)
UMBRAL = umbrales[np.argmax((2 * precision * recall / np.clip(precision + recall, 1e-12, None))[:-1])]

prob_test = modelo.predict_proba(X_test)[:, 1]

def tasas(y_real, prob, umbral):
    # TPR y FPR de la matriz de confusión
    vn, fp, fn, vp = confusion_matrix(y_real, (prob >= umbral).astype(int)).ravel()
    return vp / (vp + fn), fp / (fp + vn), (vn, fp, fn, vp)

TPR, FPR, (vn, fp, fn, vp) = tasas(y_test, prob_test, UMBRAL)
print(f'{ELEGIDO}, umbral {UMBRAL:.2f}, prueba: VP={vp}, FP={fp}, FN={fn}, VN={vn}')
print(f'TPR (sensibilidad) = {TPR:.3f}   FPR (falsas alarmas) = {FPR:.3f}')

## 3. Ganancia esperada por cliente evaluado (fórmula de Provost)

**Matriz de costo-beneficio.** La línea base es *no usar el modelo* (no hay campaña), así que solo cambian las
casillas en las que el modelo contacta a alguien (diap. 27: no contar dos veces la pérdida del cliente que se va):

| | Iba a abandonar (p) | Se iba a quedar (n) |
|---|---|---|
| **Contactado (P)** | VP: r·V − c | FP: −c |
| **No contactado (N)** | FN: 0 | VN: 0 |

**Valor esperado por cliente evaluado** (diap. 23), con el prior del banco *p(p)* = 0,15:

$$\hat a = p(p)\,[\,TPR\cdot(rV-c)\,] + p(n)\,[\,FPR\cdot(-c)\,]$$

**Fórmula simplificada** (diap. 15), por cliente contactado, con la precisión *P* como "exactitud":
$\hat a = a - (1-P)\,e$, con $a = rV - c$ (lo que deja un acierto) y $e = rV$ (lo que se deja de ganar cuando el
contactado no iba a abandonar). Se simplifica a $\hat a = P\,rV - c$, y la **precisión de equilibrio** (diap. 16) es
$P_e = 1 - a/e = c / (rV)$: por debajo de ella cada contacto pierde dinero.

In [ ]:
def valor_esperado(tpr, fpr, v=V, c=None, p=TASA_CHURN, r=R_EXITO):
    # Fórmula de Provost: ganancia esperada por cliente evaluado (COP/año)
    c = v * C_SOBRE_V if c is None else c
    return p * tpr * (r * v - c) + (1 - p) * fpr * (-c)

ev = valor_esperado(TPR, FPR, c=C)
ev_dataset = valor_esperado(TPR, FPR, c=C, p=y_test.mean())

# Fórmula simplificada, por cliente contactado
pct_contactados = TASA_CHURN * TPR + (1 - TASA_CHURN) * FPR
P_banco = TASA_CHURN * TPR / pct_contactados          # precisión esperada con el prior del banco
a_hat = P_banco * R_EXITO * V - C
P_e = C / (R_EXITO * V)

print(f'Provost: ganancia esperada = COP {ev:,.0f} por cliente evaluado y por año (prior 15 %)')
print(f'         con el prior del dataset ({y_test.mean():.1%}): COP {ev_dataset:,.0f}')
print(f'Contacta al {pct_contactados:.1%} de los clientes; precisión esperada en el banco P = {P_banco:.3f}')
print(f'Simplificada: â = COP {a_hat:,.0f} por cliente contactado  '
      f'(× {pct_contactados:.3f} contactados = COP {a_hat * pct_contactados:,.0f} por cliente evaluado)')
print(f'Precisión de equilibrio P_e = c/(r·V) = {P_e:.3f}  frente a P = {P_banco:.3f}')

**Qué vemos:**

- Con el modelo, el banco gana \~COP 13.500 por cliente evaluado y por año. La campaña contacta al 15,6 % de los
  clientes, y aproximadamente la mitad de los contactados sí iba a abandonar (precisión esperada 0,524).
- Las dos fórmulas del curso dan lo mismo. Por cliente contactado la ganancia es \~COP 86.700. Multiplicada por la
  proporción de contactados, da la misma cifra de Provost: la fórmula simplificada es Provost vista solo sobre los
  contactados.
- El modelo está muy por encima del equilibrio. Cada contacto deja dinero mientras la precisión supere *P<sub>e</sub>* =
  0,111, y el modelo tiene 0,524, unas 4,7 veces más. Esto ocurre porque un acierto vale mucho (*rV* = COP 210.000)
  frente a lo que cuesta una oferta (\~COP 23.300).
- El prior importa. Con la proporción de abandonos del dataset (20,4 %) en lugar de la del banco (15 %), la ganancia
  sube a \~COP 19.100 (+41 %). Es la pregunta de la diapositiva 28: un dataset con más abandonos que la realidad infla
  el valor si no se corrige.

## 4. Punto de equilibrio

Como en la diapositiva 20 y la Práctica 4: el beneficio mensual es *B* = clientes × â / 12, el mes de equilibrio
(ROI = 0) es $t^* = \text{Inversión} / B$ y el ROI a 12 meses es $(12B - \text{Inversión}) / \text{Inversión}$.

In [ ]:
def equilibrio(ev_cliente, inversion=INVERSION):
    b = N_CLIENTES * ev_cliente / 12
    return b, (inversion / b if b > 0 else np.inf), (12 * b - inversion) / inversion

B, t_eq, roi_12 = equilibrio(ev)
print(f'Beneficio mensual B = COP {B:,.0f}  |  ganancia anual = COP {12 * B:,.0f}')
print(f'Mes de equilibrio t* = {t_eq:.1f} meses  |  ROI a 12 meses = {roi_12:.0%}')

meses = np.arange(0, 25)
fig, ax = plt.subplots(figsize=(8, 4.2))
for v in V_RANGO:
    b, t, _ = equilibrio(valor_esperado(TPR, FPR, v=v))
    ax.plot(meses, (b * meses - INVERSION) / 1e6, marker='o', ms=3, label=f'V = COP {v / 1e3:,.0f} mil (t* = {t:.1f} meses)')
ax.axhline(0, color='black', lw=1)
ax.set_xlabel('meses desde el despliegue'); ax.set_ylabel('beneficio acumulado − inversión (millones de COP)')
ax.set_title(f'{ELEGIDO}: recuperación de la inversión (COP {INVERSION / 1e6:,.1f} millones)', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/07_punto_equilibrio.png', bbox_inches='tight'); plt.show()

**Qué vemos:** con los supuestos centrales, el banco de 20.000 clientes gana \~COP 22,6 millones al mes
(\~COP 271 millones al año). La inversión de \~COP 61 millones se recupera en **2,7 meses**, y el ROI a 12 meses es de
344 %. Incluso con el ingreso más bajo del rango (*V* = COP 500.000), la curva cruza el cero antes del cuarto mes.

## 5. Estimación paramétrica: sensibilidad a los supuestos

Los dos supuestos menos seguros son el ingreso por cliente *V* (el enunciado da un rango) y el costo por contacto *c*
(tomado de un ejemplo del curso). La tabla recalcula la ganancia por cliente y el mes de equilibrio para combinaciones
de ambos, con la misma matriz de confusión.

In [ ]:
C_RANGO = (10_000, 25_000, 50_000, 100_000)
filas = []
for v in V_RANGO:
    for c in C_RANGO:
        e = valor_esperado(TPR, FPR, v=v, c=c)
        filas.append({'V (COP/año)': v, 'c (COP/contacto)': c, 'P_e': c / (R_EXITO * v),
                      'ganancia por cliente (COP)': e, 'mes de equilibrio': equilibrio(e)[1]})
sensibilidad = pd.DataFrame(filas)
tabla = lambda col: sensibilidad.pivot(index='V (COP/año)', columns='c (COP/contacto)', values=col)
print('Ganancia esperada por cliente evaluado (COP/año):')
display(tabla('ganancia por cliente (COP)').map(lambda x: f'{x:,.0f}'))
print('Mes de equilibrio:')
display(tabla('mes de equilibrio').map(lambda x: f'{x:.1f}' if np.isfinite(x) else 'nunca'))
print(f'Precisión esperada del modelo en el banco: {P_banco:.3f}. '
      f'P_e máxima de la tabla: {sensibilidad["P_e"].max():.3f}')

**Qué vemos:**

- La ganancia es positiva en 11 de las 12 combinaciones. Solo se pierde dinero con el ingreso más bajo (COP 500.000) y
  una oferta de COP 100.000: ahí la precisión de equilibrio (0,667) supera la del modelo (0,524).
- El costo por contacto es el supuesto que más mueve el resultado. Con *V* central, la inversión se recupera en menos de
  4 meses mientras la oferta cueste hasta COP 50.000 (3,9 meses). Con COP 100.000 tarda 23,3 meses.
- La conclusión "el modelo se paga en pocos meses" se sostiene si la oferta cuesta una fracción pequeña del ingreso
  anual del cliente. Antes de desplegar hay que confirmar el costo real de la oferta.

## 6. Umbral de decisión según la ganancia

El `05` fijó el umbral por F1, que trata igual a los dos tipos de error. Aquí, como en el paso 6 de la Práctica 4, se
prueban umbrales entre 0,1 y 0,7 (la práctica usa 0,3–0,7; se amplía hacia abajo porque la precisión de
equilibrio es baja) y se elige el de **mayor ganancia en validación cruzada** (entrenamiento). La prueba solo
se usa para reportar el elegido.

In [ ]:
filas = []
for u in [UMBRAL, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7]:
    tpr_cv, fpr_cv, _ = tasas(y_train, prob_cv, u)
    tpr_t, fpr_t, _ = tasas(y_test, prob_test, u)
    filas.append({'umbral': round(u, 2), 'TPR (CV)': tpr_cv, 'FPR (CV)': fpr_cv,
                  'ganancia por cliente, CV (COP)': valor_esperado(tpr_cv, fpr_cv),
                  'ganancia por cliente, prueba (COP)': valor_esperado(tpr_t, fpr_t)})
tabla_umbral = pd.DataFrame(filas).sort_values('umbral').set_index('umbral')
display(tabla_umbral[['TPR (CV)', 'FPR (CV)', 'ganancia por cliente, CV (COP)']])

U_GANANCIA = tabla_umbral['ganancia por cliente, CV (COP)'].idxmax()
e_u = tabla_umbral.loc[U_GANANCIA, 'ganancia por cliente, prueba (COP)']
print(f'Contactados con el umbral elegido (prior del banco): '
      f'{TASA_CHURN * tasas(y_test, prob_test, U_GANANCIA)[0] + (1 - TASA_CHURN) * tasas(y_test, prob_test, U_GANANCIA)[1]:.1%}')
print(f'Umbral de mayor ganancia en CV: {U_GANANCIA:.2f}. En prueba: COP {e_u:,.0f} por cliente, '
      f'equilibrio en {equilibrio(e_u)[1]:.1f} meses (umbral del 05: COP {ev:,.0f}, {t_eq:.1f} meses)')

**Qué vemos:**

- La ganancia en validación es máxima con un umbral de 0,15 y casi plana entre 0,15 y 0,25 (\~COP 15.700–16.600 por
  cliente). Con 0,50 baja a \~COP 10.900. El umbral por F1 del `05` (0,33) deja dinero sobre la mesa porque penaliza
  igual una falsa alarma (cuesta \~COP 23.300) que un abandono no detectado (deja de ganar *rV* = COP 210.000).
- En prueba, el umbral 0,15 da COP 16.035 por cliente, frente a COP 13.539 con 0,33 (+18 %), y el equilibrio pasa de 2,7
  a 2,3 meses.
- El costo operativo es mayor. Con 0,15 se contacta al 40,4 % de la base (\~8.000 clientes al año), frente al 15,6 %. Si
  el banco no tiene capacidad para una campaña tan grande, o si la oferta resulta más cara, conviene un umbral más alto.
  El umbral óptimo depende de *c*, así que se debe recalcular con el costo real.

## 7. Limitaciones

**De la construcción del modelo:**

- **Datos de otro mercado.** El modelo se entrenó con clientes europeos (un dataset del curso con errores de calidad
  introducidos a propósito) y se aplica a un banco colombiano. No sabemos si la sensibilidad y la tasa de falsas
  alarmas se mantienen. Hace falta un piloto con clientes reales.
- **Una sola partición de prueba** (\~2.000 clientes). La sensibilidad y la tasa de falsas alarmas tienen
  incertidumbre (intervalos en el `05`), y esa incertidumbre no se propagó al valor.
- **Sesgos detectados en el `05`.** El modelo subestima el riesgo en Alemania y en mujeres, así que el valor real en
  esos grupos es menor. La Práctica 4 sugiere calcular el valor por segmento, y aquí no se hizo.
- **Cambio de prior.** Pasar del 20 % al 15 % de abandonos supone que la sensibilidad y la tasa de falsas alarmas no
  cambian con la tasa de abandono.

**Del valor que genera:**

- **Supuestos del curso, no del mercado.** El costo por contacto y la inversión vienen de ejemplos del curso, no de
  cotizaciones reales. La inversión no incluye costos recurrentes (monitoreo, reentrenamiento, operación de la
  campaña), que la Práctica 4 señala como faltantes.
- ***V* es ingreso, no margen**, y solo cuenta un año de permanencia. Lo primero sobrestima la ganancia; lo segundo
  puede subestimarla si el cliente retenido se queda más tiempo.
- **Éxito de la oferta.** Se supone un 30 % de éxito, igual para todos. Tampoco se modela el efecto de contactar a
  quien no pensaba irse (molestia, o un descuento que se regala).
- **Correlación, no causalidad.** El modelo identifica a quién contactar, no por qué la oferta funcionaría. El 30 % solo
  se puede validar con un experimento (grupo de control sin oferta).

## 8. Conclusiones

- **Ganancia esperada por estimación:** con los supuestos centrales, \~COP 13.500 por cliente evaluado y por año (fórmula
  de Provost, prior del 15 %). Equivale a \~COP 271 millones al año para un banco de 20.000 clientes.
- **Punto de equilibrio:** la inversión de \~COP 61 millones se recupera en \~2,7 meses. En términos de calidad, el
  modelo deja ganancia mientras su precisión supere 0,111, y la suya es 0,524.
- El resultado cambia poco con el ingreso por cliente y mucho con el costo de la oferta. Solo se pierde dinero si la
  oferta cuesta mucho frente al ingreso (COP 100.000 con *V* = COP 500.000).
- El umbral se debe fijar por ganancia, no por F1. Bajarlo a \~0,15 aumenta la ganancia \~18 %, a cambio de contactar
  al 40 % de la base.
- **Recomendación:** antes de desplegar, hacer un piloto que mida el costo real por contacto y la tasa de éxito real
  de la oferta, con un grupo de control. Esos dos números deciden el umbral y el valor.